# RAG Experimentation & Analysis

This notebook demonstrates the resume RAG pipeline, evaluates retrieval quality,
and benchmarks latency for the profile matching assignment.

In [ ]:
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Add project root to path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from config import JOB_DESCRIPTIONS_DIR, RESUMES_DIR
from job_matcher import JobMatcher
from resume_rag import ResumeRAG
from tools.resume_loader import list_resumes, load_all_resumes

## 1. Dataset Overview

In [ ]:
listing = list_resumes(RESUMES_DIR)
print(f"Resume count: {listing['count']}")
print(f"Directory: {listing['directory']}")

batch = load_all_resumes(RESUMES_DIR)
print(f"Loaded: {batch['loaded_count']}, Failed: {batch['failed_count']}")

rows = []
for doc in batch["documents"]:
    m = doc["metadata"]
    rows.append({
        "name": m["name"],
        "experience_years": m["experience_years"],
        "skill_count": len(m["skills"]),
        "sections": ", ".join(m["section_names"]),
    })
df_meta = pd.DataFrame(rows)
df_meta.head(10)

## 2. Build Vector Index

In [ ]:
rag = ResumeRAG()
rag.reset_index()

index_start = time.perf_counter()
index_result = rag.index_resumes(RESUMES_DIR)
index_latency = time.perf_counter() - index_start

print(json.dumps(index_result, indent=2))
print(f"\nTotal indexing time: {index_latency:.2f}s")
rag.get_stats()

## 3. Semantic Search Demo

In [ ]:
query = "Senior Python machine learning engineer with AWS and Docker experience"
search_result = rag.query(query, top_k=5)

for i, hit in enumerate(search_result["results"], 1):
    meta = hit["metadata"]
    print(f"{i}. {meta['candidate_name']} | section={meta['section']} | sim={hit['similarity']}")
    print(f"   {hit['text'][:120]}...\n")

## 4. Job Matching (Hybrid Search)

In [ ]:
matcher = JobMatcher(rag)
jd_path = JOB_DESCRIPTIONS_DIR / "senior_python_ml_engineer.json"

match_start = time.perf_counter()
match_result = matcher.match_job_file(jd_path)
match_latency = time.perf_counter() - match_start

print(f"Job: {match_result.get('job_title')}")
print(f"Match latency: {match_latency:.3f}s\n")

for m in match_result["top_matches"][:5]:
    print(f"{m['candidate_name']:20s} score={m['match_score']} skills={m['matched_skills'][:4]}")

## 5. Retrieval Accuracy Evaluation

We define expected top candidates per job (based on role profiles in synthetic data)
and measure whether they appear in the top-K results.

In [ ]:
# Ground truth: job -> keywords that should match candidate role profiles
EVAL_CASES = {
    "senior_python_ml_engineer.json": ["ml engineer", "machine learning", "tensorflow", "pytorch"],
    "full_stack_developer.json": ["full stack", "react", "node.js"],
    "devops_engineer.json": ["devops", "kubernetes", "terraform"],
    "data_scientist.json": ["data scientist", "pandas", "scikit"],
    "nlp_engineer.json": ["nlp", "pytorch", "fastapi"],
    "backend_java_developer.json": ["java", "spring", "microservices"],
}

K = 10
eval_rows = []

for job_file, keywords in EVAL_CASES.items():
    result = matcher.match_job_file(JOB_DESCRIPTIONS_DIR / job_file, top_k=K)
    top_names = [m["candidate_name"].lower() for m in result.get("top_matches", [])]
    top_text = " ".join(
        " ".join(m.get("relevant_excerpts", [])) for m in result.get("top_matches", [])
    ).lower()

    hits = sum(1 for kw in keywords if any(kw in n or kw in top_text for n in top_names))
    accuracy = hits / len(keywords) if keywords else 0

    eval_rows.append({
        "job": job_file.replace(".json", ""),
        "keyword_hits": hits,
        "keyword_total": len(keywords),
        "accuracy": round(accuracy * 100, 1),
        "top_candidate": result["top_matches"][0]["candidate_name"] if result.get("top_matches") else "N/A",
        "top_score": result["top_matches"][0]["match_score"] if result.get("top_matches") else 0,
    })

df_eval = pd.DataFrame(eval_rows)
df_eval

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(df_eval["job"], df_eval["accuracy"], color="steelblue")
ax.set_ylabel("Keyword Recall (%)")
ax.set_xlabel("Job Description")
ax.set_title("Retrieval Accuracy by Job (Top-10 Hybrid Search)")
ax.set_ylim(0, 100)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

print(f"Mean accuracy: {df_eval['accuracy'].mean():.1f}%")

## 6. Latency Benchmarks

In [ ]:
latencies = {"index_build": [], "semantic_query": [], "job_match": []}
N = 5

for _ in range(N):
    t0 = time.perf_counter()
    rag.query("Python developer with cloud experience", top_k=10)
    latencies["semantic_query"].append(time.perf_counter() - t0)

    t0 = time.perf_counter()
    matcher.match_job_file(jd_path)
    latencies["job_match"].append(time.perf_counter() - t0)

latencies["index_build"] = [index_latency]

df_latency = pd.DataFrame({
    "operation": ["Index Build (34 resumes)", "Semantic Query", "Job Match"],
    "mean_s": [
        index_latency,
        sum(latencies["semantic_query"]) / N,
        sum(latencies["job_match"]) / N,
    ],
    "min_s": [
        index_latency,
        min(latencies["semantic_query"]),
        min(latencies["job_match"]),
    ],
    "max_s": [
        index_latency,
        max(latencies["semantic_query"]),
        max(latencies["job_match"]),
    ],
})
df_latency

## 7. Hybrid vs Semantic-Only Comparison

In [ ]:
hybrid = matcher.match_job_file(jd_path, semantic_weight=0.65, keyword_weight=0.35)
semantic_only = matcher.match_job_file(jd_path, semantic_weight=1.0, keyword_weight=0.0)

comparison = pd.DataFrame({
    "rank": range(1, 6),
    "hybrid_candidate": [m["candidate_name"] for m in hybrid["top_matches"][:5]],
    "hybrid_score": [m["match_score"] for m in hybrid["top_matches"][:5]],
    "semantic_candidate": [m["candidate_name"] for m in semantic_only["top_matches"][:5]],
    "semantic_score": [m["match_score"] for m in semantic_only["top_matches"][:5]],
})
comparison

## 8. Error Handling Demo

In [ ]:
from tools.resume_loader import validate_file, load_resume

# Invalid file type
print("Invalid extension:", validate_file("README.md"))

# Missing file
print("Missing file:", validate_file("resumes/nonexistent.txt"))

# Empty query
print("Empty query:", rag.query(""))